# Laboratorio 01 — Exploración libre con PySpark

**Semana:** 02 | **Actividad de referencia:** Actividad 01  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Este laboratorio es tu espacio para aplicar lo aprendido en la Actividad 01 sobre un dataset **de tu elección**. No hay respuestas correctas predefinidas — el valor está en demostrar que puedes explorar datos desconocidos de forma sistemática y extraer conclusiones de negocio.

Entrega esperada: este notebook completo con todas las celdas ejecutadas y las celdas markdown respondidas.

## Parte 1 — Descripción del dataset

Antes de escribir código, documenta aquí tu dataset. Responde cada punto:

1. **Nombre y fuente:** ¿Cómo se llama el dataset y de dónde lo obtuviste? (incluye URL)
2. **Dominio:** ¿Qué problema o área describe? (finanzas, salud, deporte, transporte...)
3. **¿Por qué lo elegiste?** Explica qué te pareció interesante o qué pregunta de negocio quieres responder con él.
4. **Número aproximado de filas y archivos:** ¿Cuántos registros tiene? ¿Es un solo archivo o varios?
5. **Preguntas de negocio:** Lista al menos 3 preguntas que crees que se pueden responder con este dataset. (Las responderás en la Parte 5.)

> Fuentes sugeridas si aún no tienes dataset:
> - [Kaggle Datasets](https://www.kaggle.com/datasets)
> - [datos.gob.es](https://datos.gob.es)
> - [UCI Machine Learning Repository](https://archive.ics.uci.edu/datasets)
> - [Google Dataset Search](https://datasetsearch.research.google.com)

**Escribe tu respuesta aquí:**

## Parte 2 — Carga y perfil técnico del dataset

Carga el archivo en un DataFrame PySpark y ejecuta el perfil técnico completo. Para cada sección, añade una celda markdown con tus observaciones.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, isnan, count as spark_count, sum as spark_sum

# Ajusta VOL y el nombre del archivo a tu dataset
VOL = "/Volumes/workspace/default/week_2"  # cambia si usas un volumen distinto
ARCHIVO = "tu_archivo.csv"                 # cambia por el nombre real

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOL}/{ARCHIVO}")

print(f"Filas: {df.count():,}")
print(f"Columnas: {len(df.columns)}")

In [ ]:
# Schema — tipos de cada columna
df.printSchema()

**Observaciones del schema:** ¿`inferSchema` interpretó correctamente los tipos? ¿Qué columnas tuviste que castear manualmente?

In [ ]:
# Vista rápida de los primeros registros
df.show(10, truncate=False)

In [ ]:
# Estadísticas descriptivas de columnas numéricas
df.describe().show(truncate=False)

**Observaciones del describe:** ¿Hay valores mínimos o máximos inesperados? ¿Alguna columna con desviación estándar muy alta?

In [ ]:
# Análisis de nulos y vacíos por columna
total = df.count()
numeric_types = {"double", "float", "long", "integer", "short", "byte"}

perfil = df.select([
    spark_sum(
        when(
            col(c).isNull() |
            (isnan(col(c)) if df.schema[c].dataType.typeName() in numeric_types else F.lit(False)) |
            (col(c).cast("string") == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
])

nulos = perfil.collect()[0].asDict()
print(f"{'Columna':<35} {'Nulos':>8} {'% Nulos':>10}")
print("-" * 56)
for c, n in sorted(nulos.items(), key=lambda x: -x[1]):
    print(f"{c:<35} {n:>8,} {n/total*100:>9.1f}%")

**Observaciones de nulos:** ¿Qué columnas tienen más nulos? ¿Crees que son aleatorios o tienen un patrón? ¿Qué harías en un pipeline real: eliminar filas, imputar, o mantener?

In [ ]:
# Cardinalidad de columnas categóricas (string)
string_cols = [f.name for f in df.schema.fields if f.dataType.typeName() == "string"]

print(f"{'Columna':<35} {'Valores únicos':>15}")
print("-" * 52)
for c in string_cols:
    card = df.select(c).distinct().count()
    print(f"{c:<35} {card:>15,}")

**Observaciones de cardinalidad:** ¿Qué columnas tienen baja cardinalidad (buenos candidatos para `groupBy`)? ¿Cuáles tienen alta cardinalidad (posibles IDs o texto libre)?

## Parte 3 — Transformaciones de la Actividad 01

Aplica al menos **4 transformaciones** de las vistas en la Actividad 01. Para cada una:
- Escribe el código
- Explica en markdown qué hace y por qué tiene sentido para tu dataset

Transformaciones disponibles (elige las que apliquen a tu dataset):
- Cast de tipos (`cast`)
- Limpieza de strings (`regexp_replace`, `trim`, `lower`, `upper`)
- Extracción de componentes de fecha (`year`, `month`, `dayofweek`, `hour`...)
- Creación de columnas derivadas (`withColumn`)
- Filtros (`filter` / `where`)
- Renombrado de columnas (`withColumnRenamed`)
- Selección de columnas (`select`)
- Eliminación de duplicados (`dropDuplicates`)

In [ ]:
# Transformación 1 — describe qué hace en el comentario

**Explicación transformación 1:**

In [ ]:
# Transformación 2

**Explicación transformación 2:**

In [ ]:
# Transformación 3

**Explicación transformación 3:**

In [ ]:
# Transformación 4

**Explicación transformación 4:**

## Parte 4 — Evaluación lazy y plan de ejecución

Construye una cadena de al menos 3 transformaciones encadenadas (sin ejecutar ninguna acción todavía). Luego usa `df.explain()` para ver el plan lógico y físico.

In [ ]:
# Cadena de transformaciones — sin show() ni count() aún
df_plan = (
    df
    # .filter(...)
    # .withColumn(...)
    # .groupBy(...).agg(...)
)

df_plan.explain()

**¿Qué está haciendo Spark en ese plan antes de materializar los datos?** Describe en tus propias palabras qué operaciones aparecen y en qué orden se ejecutarán.

In [ ]:
# Ahora ejecuta la acción y muestra el resultado
df_plan.show(10, truncate=False)

## Parte 5 — Análisis de negocio

Responde las 3 preguntas de negocio que planteaste en la Parte 1. Cada respuesta debe tener:
- Un bloque de código PySpark
- Una celda markdown con tu conclusión en lenguaje natural (como si se la explicaras a alguien sin conocimientos técnicos)

In [ ]:
# Pregunta 1:

**Conclusión pregunta 1:**

In [ ]:
# Pregunta 2:

**Conclusión pregunta 2:**

In [ ]:
# Pregunta 3:

**Conclusión pregunta 3:**

## Parte 6 — Reflexión final

Responde en esta celda:

1. ¿Qué fue lo más sorprendente que encontraste en los datos?
2. ¿Qué transformación te resultó más difícil de aplicar a tu dataset y por qué?
3. ¿Qué pregunta de negocio adicional te gustaría responder si tuvieras más tiempo?
4. ¿En qué se diferencia explorar este dataset con PySpark respecto a hacerlo con pandas o Excel?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_02/laboratorios/lab_01_exploracion.ipynb semana_02/laboratorios/<tu-nombre>/lab_01_exploracion.ipynb

git add semana_02/laboratorios/<tu-nombre>/lab_01_exploracion.ipynb
git commit -m "lab: semana02 lab01 exploracion <nombre-dataset> - <tu-nombre>"
git push origin develop
```